In [1]:
import pandas as pd
import numpy as np
import re
from io import StringIO

In [2]:
file_path_1 = r'C:\Users\482525\Desktop\敗血症資料\2024\2024Sepsis00106.csv'
file_path_2 = r'C:\Users\482525\Desktop\敗血症資料\2025\2025Sepsis00106.csv'

with open(file_path_1, 'r', encoding='cp950', errors='ignore') as f1:
    content1 = f1.read()
with open(file_path_2, 'r', encoding='cp950', errors='ignore') as f2:
    content2 = f2.read()
    
df2024 = pd.read_csv(StringIO(content1), low_memory=False, dtype={'ORDERTIME': str, 'GATHERTIME': str, 'VERIFYTIME': str})
df2025 = pd.read_csv(StringIO(content2), low_memory=False, dtype={'ORDERTIME': str, 'GATHERTIME': str, 'VERIFYTIME': str})

len(df2024), len(df2025)

(1048575, 993767)

In [3]:
df2024['ORDERTIME'] = pd.to_datetime(df2024['ORDERTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2024['GATHERTIME'] = pd.to_datetime(df2024['GATHERTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2024['VERIFYTIME'] = pd.to_datetime(df2024['VERIFYTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2025['ORDERTIME'] = pd.to_datetime(df2025['ORDERTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2025['GATHERTIME'] = pd.to_datetime(df2025['GATHERTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2025['VERIFYTIME'] = pd.to_datetime(df2025['VERIFYTIME'], format='%Y%m%d%H%M%S', errors='coerce')

table6 = pd.concat([df2024, df2025], ignore_index=True)
print(len(table6))

2042342


In [4]:
table6 = table6.dropna(how='all')
table6 = table6[table6['ORDERTIME'].notna()]
table6 = table6.loc[:, ~table6.columns.str.contains('^Unnamed')]

In [5]:
table6

,DIVISIONNO,ACCOUNTNO,IPDACCOUNTNO,ORDERNO,LABITEMNO,ORDERNAME,ITEMNAME,RESULTVALUE,ORDERTIME,GATHERTIME,VERIFYTIME
1048575,001,I11400000003,NaN,L4000200,L40201,尿沉渣Urine Sediment,Microscopic RBC,0-2,2025-01-01 00:31:00,2025-01-01 01:01:09,2025-01-01 01:03:01
1048576,001,I11400000003,NaN,L4000200,L40202,尿沉渣Urine Sediment,Microscopic WBC,20-29,2025-01-01 00:31:00,2025-01-01 01:01:09,2025-01-01 01:03:01
1048577,001,I11400000003,NaN,L4000200,L40203,尿沉渣Urine Sediment,Ep.cell,0-5,2025-01-01 00:31:00,2025-01-01 01:01:09,2025-01-01 01:03:01
1048578,001,I11400000003,NaN,L4000200,L40204,尿沉渣Urine Sediment,Cast(1),-,2025-01-01 00:31:00,2025-01-01 01:01:09,2025-01-01 01:03:01
1048579,001,I11400000003,NaN,L4000200,L40205,尿沉渣Urine Sediment,Crystal (1),-,2025-01-01 00:31:00,2025-01-01 01:01:09,2025-01-01 01:03:01
...,...,...,...,...,...,...,...,...,...,...,...
2042336,001,I11400060739,NaN,L6001100,L61101,GOT,GOT,18,2025-12-31 23:50:00,2025-12-31 23:05:08,2026-01-01 00:02:05
2042337,001,I11400060739,NaN,L6001700,L61701,Creatinine,Creatinine,0.54,2025-12-31 23:50:00,2025-12-31 23:05:08,2026-01-01 00:02:05
2042338,001,I11400060739,NaN,L6001700,L61702,Creatinine,eGFR(MDRD計算公式，僅供參考),126.3,2025-12-31 23:50:00,2025-12-31 23:05:08,2026-01-01 00:02:05
2042339,001,I11400060739,NaN,L6001700,L61703,Creatinine,eGFR(CKD-EPI計算公式，僅供參考),120.8,2025-12-31 23:50:00,2025-12-31 23:05:08,2026-01-01 00:02:05


In [6]:
len(table6),len(table6['ACCOUNTNO'].unique())

(993766, 27112)

In [8]:
table6 = table6[['ACCOUNTNO','ITEMNAME', 'RESULTVALUE']]

In [28]:
# raw_text = """'
# '人工閱片', 'Neutrophil Seg.', 'Lymphocyte', 'Monocyte', 'Eosinophil',
#        'Basophil', 'Absolute Neutrophil count', 'Hb', 'Ht', 'RBC', 'RDW',
#        'WBC', 'MCV', 'MCH', 'MCHC', 'PLT', 'Na', 'K', 'Glucose', 'GPT',
# """

# # 簡單清理並轉成一維清單
# items = [i.strip().strip("'") for i in raw_text.replace('\n', ',').split(',') if i.strip()]

In [9]:
wbc_df = table6[table6['ITEMNAME'] == 'WBC'].copy()

wbc_df['RESULTVALUE_wbc'] = (wbc_df['RESULTVALUE'].str.replace('<', '', regex=False).str.replace('>', '', regex=False))
wbc_df['RESULTVALUE_wbc'] = pd.to_numeric(wbc_df['RESULTVALUE_wbc'], errors='coerce')

wbc_pivot = wbc_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_wbc', aggfunc='max'
                              ).rename(columns={'RESULTVALUE_wbc': 'WBC'})

print(wbc_pivot.shape)

(23425, 1)


In [10]:
crp_df = table6[table6['ITEMNAME'] == 'CRP'].copy()
crp_df['with_below'] = crp_df['RESULTVALUE'].str.contains('<', na=False)

crp_df['clean_values'] = crp_df['RESULTVALUE'].str.replace('<', '', regex=False)
crp_df['clean_values'] = pd.to_numeric(crp_df['clean_values'], errors='coerce')

# 有<0.1這種就/2
crp_df['RESULTVALUE_crp'] = np.where(crp_df['with_below'], crp_df['clean_values'] / 2, crp_df['clean_values'])

crp_pivot = crp_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_crp', aggfunc='max').rename(columns={'RESULTVALUE_crp': 'CRP'})
print(crp_pivot.shape)

(13342, 1)


In [11]:
lym_df = table6[table6['ITEMNAME'] == 'Lymphocyte'].copy()
lym_df['RESULTVALUE_lym'] = pd.to_numeric(lym_df['RESULTVALUE'], errors='coerce')

lym_pivot = lym_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_lym', aggfunc='max').rename(columns={'RESULTVALUE_lym': 'Lymphocyte'})
print(lym_pivot.shape)

(23298, 1)


In [12]:
# neu_df = table6[table6['ITEMNAME'] == 'Absolute Neutrophil count'].copy()
# neu_df['RESULTVALUE_neu'] = pd.to_numeric(neu_df['RESULTVALUE'], errors='coerce')

# neu_pivot = neu_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_neu', aggfunc='max').rename(columns={'RESULTVALUE_neu': 'Absolute Neutrophil count'})
# print(neu_pivot.shape)

In [26]:
neu_df = table6[table6['ITEMNAME'].str.contains('Neutrophil', na=False)]
neu_df['ITEMNAME'].unique()

array(['Neutrophil Seg.', 'Absolute Neutrophil count', 'Neutrophil',
       'Neutrophil Band'], dtype=object)

In [13]:
# neu_pivot

In [33]:
igg_df = table6[table6['ITEMNAME'].str.contains('RBC', case=False, na=False)]

In [34]:
igg_df['ITEMNAME'].unique()

array(['Microscopic RBC', 'RBC', 'Stool -Microscopic RBC', 'Dymor.RBC',
       'RBC count', 'NRBC', 'RBC Morphology', 'RBC Number',
       'RBC morphology Fresh：Old', 'RBC morphology'], dtype=object)

In [16]:
merge_df1 = wbc_pivot.merge(lym_pivot, on='ACCOUNTNO', how='left')
table6_clear = merge_df1.merge(crp_pivot, on='ACCOUNTNO', how='left')

In [17]:
table6_clear = table6_clear.reset_index()

In [18]:
table6_clear.to_csv('table6_clear.csv', index=False)